In [2]:
import os
import getpass
import pandas as pd
import faiss
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_mistralai import ChatMistralAI

#from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_classic.chains import RetrievalQA

In [3]:
# ==========================================
# Chargement, Indexation FAISS & Maintien des ID
# ==========================================

print("--- Chargement et Indexation FAISS ---")

# Charge le fichier CSV contenant les événements
# Assurez-vous que le fichier 'evenement.csv' est bien dans votre répertoire
df = pd.read_csv("./data/evenement.csv")


# Converti chaque ligne du DataFrame en Document LangChain
# On stocke l'ID et les détails dans les métadonnées pour garder la liaison directe
documents = []
for _, row in df.iterrows():
    # Le texte à vectoriser (colonne demandée)
    texte_vecteur = str(row['texte_a_vectoriser'])
    
    # Les métadonnées pour retrouver les détails complets dans le CSV plus tard
    metadata = {
        "id": row['id'],
        "titre": row.get('titre', ''),
        "lieu": row.get('lieu', ''),
        "ville": row.get('ville', ''),
        "url": row.get('url', ''),
        "date": row.get('date', '')
    }
    
    documents.append(Document(page_content=texte_vecteur, metadata=metadata))

print(f"{len(documents)} documents chargés depuis 'evenement.csv'.")


--- Chargement et Indexation FAISS ---
229 documents chargés depuis 'evenement.csv'.


In [4]:
# 3. Initialiser le modèle d'embedding (multilingue)
embeddings_model = HuggingFaceEmbeddings(
    model_name="paraphrase-multilingual-MiniLM-L12-v2"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [5]:
# 1. Define your dimensions and configure the HNSW index
dimension = 384  # Dimension des vecteurs pour le modèle MiniLM
index = faiss.IndexHNSWFlat(dimension, 32)
index.hnsw.efConstruction = 64
index.hnsw.efSearch = 32

In [6]:
# 2. Extract texts and metadatas from your documents
texts = [doc.page_content for doc in documents]
metadatas = [doc.metadata for doc in documents]

In [7]:
# 3. Embed the texts
embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings = embeddings_model.embed_documents(texts)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
# 4. Create the docstore and index mapping required by LangChain
docstore = InMemoryDocstore({
    str(i): doc for i, doc in enumerate(documents)
})
index_to_docstore_id = {i: str(i) for i in range(len(documents))}

In [9]:
# 5. Instantiate FAISS directly using your custom index
db_faiss = FAISS(
    embedding_function=embeddings_model,
    index=index,
    docstore=docstore,
    index_to_docstore_id=index_to_docstore_id
)

In [10]:
# 6. Add the embeddings to the vector store index
db_faiss.add_embeddings(
    text_embeddings=list(zip(texts, embeddings)),
    metadatas=metadatas
)

['f1906156-c233-4bf8-9650-f87567c7f9d4',
 '6c0b1575-5d8f-4bdb-b35e-7a5afa77f997',
 '8d549e59-83bc-4ad6-bc4c-93b820a53ca0',
 '48dc810d-ec7c-4c6f-9a7e-e121829b1f1f',
 '5085d5ea-5707-42d3-860e-1f0b557da363',
 '421f2331-81e6-47c0-99db-1daf107b432e',
 '9f42edc0-249c-43c9-9061-9b03b3663022',
 '2f735ab1-7fe5-416d-b24f-6921bf0d1611',
 '6008d0ab-d57f-430c-b7eb-fec50a59c4fe',
 'f4f0abda-46ac-4660-a21e-c298ea32f810',
 '0d6971a0-2b41-46bf-94fc-6c36e841d744',
 '4de76ac0-47e4-4e72-b4f5-90af5a95f842',
 'a87e3a45-1ff2-4d05-ad29-94bd8496d767',
 '6bc80f7b-5f52-454b-bb0b-e82993aa5e6b',
 'f6c911bb-5a86-41d0-a692-867a7aa5716b',
 '68cda62c-2909-4366-9153-a66c604e4481',
 '5cb69869-b55f-4fd4-a6ad-99000a9ddf6e',
 'c4f24989-1834-4d81-80fc-2aa8f109d369',
 'b9f3f0d9-a9ad-41e1-860a-75b61d36983e',
 '95c33aab-db1f-446d-ae7d-fa7459317500',
 '16c37547-69b2-4fe3-9429-e9eebbea9ebd',
 '14cfd955-fe0e-4751-a23b-55f43e6faa67',
 '06bcf8be-61bb-49cd-be03-06802d84e2a0',
 '485b5c6a-dd0d-47b7-b4f4-5cd5015e6bbb',
 '6bc893b1-d767-

In [11]:
# Sauvegarde l'index localement pour le réutiliser si besoin
db_faiss.save_local("./data/index_openagenda_hnsw")
print("Index FAISS optimisé (HNSW) créé et sauvegardé avec succès !\n")

Index FAISS optimisé (HNSW) créé et sauvegardé avec succès !



# ==========================================
#  Chatbot  avec l'API Mistral
# ==========================================

In [38]:
print("---  Configuration du Chatbot Mistral ---")



api_key = os.getenv("MISTRAL_API_KEY")
if not api_key:
    raise ValueError(
        "La clé API  n'est pas définie dans l'environnement."
    )

---  Configuration du Chatbot Mistral ---


In [43]:

#  Initialiser le modèle de langage Mistral
llm = ChatMistralAI(
    model="mistral-tiny",  # Modèle rapide et performant pour le RAG
    temperature=0.3                # Température basse pour privilégier la précision factuelle
)

# 3. Transformer la base FAISS en récupérateur (retriever)
# On demande par exemple de récupérer les 3 événements les plus pertinents
retriever = db_faiss.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# 4. Créer la chaîne de questions/réponses (RetrievalQA)
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

print("Chatbot Mistral prêt à formuler des recommandations personnalisées !\n")

Chatbot Mistral prêt à formuler des recommandations personnalisées !



In [44]:
# ==========================================
# EXEMPLE D'UTILISATION DU CHATBOT
# ==========================================

# Question utilisateur en langage naturel
question_utilisateur = "Je cherche une exposition artistique ou culturelle originale à faire à Paris dans les prochains jours. Que me conseilles-tu ?"

print(f"Question posée : « {question_utilisateur} »\n")

Question posée : « Je cherche une exposition artistique ou culturelle originale à faire à Paris dans les prochains jours. Que me conseilles-tu ? »



In [45]:
# Exécution de la chaîne RAG
reponse = qa_chain.invoke({"query": question_utilisateur})

In [46]:



print("=== Réponse personnalisée de Mistral ===")
print(reponse["result"])

print("\n=== Événements sources utilisés (avec correspondance ID du CSV) ===")
for i, doc in enumerate(reponse["source_documents"]):
    # Grâce aux métadonnées, on fait le lien direct avec l'ID du fichier CSV d'origine
    print(f"{i+1}. [ID CSV : {doc.metadata['id']}] {doc.metadata['titre']}")
    print(f"   Lieu : {doc.metadata['lieu']} ({doc.metadata['ville']})")
    print(f"   Lien : {doc.metadata['url']}\n")

=== Réponse personnalisée de Mistral ===
D'après les informations disponibles, voici deux événements culturels originaux à Paris dans les prochains jours :

1. **"Explore le Monde à travers Paris"** :
   - **Description** : Un parcours inédit pour découvrir le monde à travers Paris, en partenariat avec le pass Culture.
   - **Lieu** : Institut suédois (11 rue Payenne).
   - **Période** : **Samedi 3 octobre à 15h00** (proche de la date actuelle, donc idéal si vous cherchez une activité pour ces prochains jours).

2. **"BISFF in Paris - Conférence : L’IA et le court-métrage coréen"** :
   - **Description** : Une conférence intéressante qui explore l'intersection entre l'innovation technologique coréenne et l'art cinématographique français, avec une réflexion sur l'impact de l'IA dans le cinéma.
   - **Lieu** : Centre Culturel Coréen (20 Rue La Boétie, 75008 Paris).
   - **Période** : **Lundi 12 octobre à 19h00** (un peu plus loin dans le temps, mais une option très originale et intellect

In [48]:
print("=" * 60)
print(" 🤖  RÉPONSE PERSONNALISÉE DE MISTRAL ")
print("=" * 60)
print(reponse["result"].strip())
print("\n" + "-" * 60)

print(" 📚  ÉVÉNEMENTS SOURCES UTILISÉS (RAG) ")
print("-" * 60)

for i, doc in enumerate(reponse["source_documents"], start=1):
    # Affichage propre avec des encarts et des emojis
    print(f"[{i}] 📌 {doc.metadata.get('titre', 'Titre non disponible')}")
    print(f"    📍 Lieu   : {doc.metadata.get('lieu', 'Lieu non spécifié')} ({doc.metadata.get('ville', 'Paris')})")
    
    url = doc.metadata.get('url')
    if url:
        print(f"    🔗 Lien   : {url}")
    else:
        print("    🔗 Lien   : Non disponible")
        
    print()  # Ligne vide pour aérer entre chaque source

print("=" * 60)

 🤖  RÉPONSE PERSONNALISÉE DE MISTRAL 
D'après les informations disponibles, voici deux événements culturels originaux à Paris dans les prochains jours :

1. **"Explore le Monde à travers Paris"** :
   - **Description** : Un parcours inédit pour découvrir le monde à travers Paris, en partenariat avec le pass Culture.
   - **Lieu** : Institut suédois (11 rue Payenne).
   - **Période** : **Samedi 3 octobre à 15h00** (proche de la date actuelle, donc idéal si vous cherchez une activité pour ces prochains jours).

2. **"BISFF in Paris - Conférence : L’IA et le court-métrage coréen"** :
   - **Description** : Une conférence intéressante qui explore l'intersection entre l'innovation technologique coréenne et l'art cinématographique français, avec une réflexion sur l'impact de l'IA dans le cinéma.
   - **Lieu** : Centre Culturel Coréen (20 Rue La Boétie, 75008 Paris).
   - **Période** : **Lundi 12 octobre à 19h00** (un peu plus loin dans le temps, mais une option très originale et intellectuel

In [50]:
print("=" * 60)
print(" 🤖  RÉPONSE PERSONNALISÉE DE MISTRAL ")
print("=" * 60)
print(reponse["result"].strip())
print("\n" + "-" * 60)

print(" 📚  ÉVÉNEMENTS SOURCES UTILISÉS (RAG) ")
print("-" * 60)

for i, doc in enumerate(reponse["source_documents"], start=1):
    # Fonction locale pour nettoyer les valeurs "nan" ou vides
    def clean_meta(key, default="Non spécifié"):
        val = doc.metadata.get(key)
        # Si la valeur est None, vide, ou la chaîne "nan", on renvoie le défaut
        if val is None or str(val).strip().lower() in ["", "nan", "none"]:
            return default
        return str(val).strip()

    titre = clean_meta('titre', 'Titre non disponible')
    id_csv = clean_meta('id', 'N/A')
    lieu = clean_meta('lieu', 'Lieu non spécifié')
    ville = clean_meta('ville', '')
    url = clean_meta('url', '')

    # Affichage propre
    print(f"[{i}] 📌 {titre}")

    
    # Gérer l'affichage du lieu et de la ville proprement
    if ville and ville != "Non spécifié":
        print(f"    📍 Lieu   : {lieu} ({ville})")
    else:
        print(f"    📍 Lieu   : {lieu}")
        
    if url and url != "Non spécifié":
        print(f"    🔗 Lien   : {url}")
    else:
        print("    🔗 Lien   : Non disponible")
        
    print()  # Ligne vide pour aérer

print("=" * 60)

 🤖  RÉPONSE PERSONNALISÉE DE MISTRAL 
D'après les informations disponibles, voici deux événements culturels originaux à Paris dans les prochains jours :

1. **"Explore le Monde à travers Paris"** :
   - **Description** : Un parcours inédit pour découvrir le monde à travers Paris, en partenariat avec le pass Culture.
   - **Lieu** : Institut suédois (11 rue Payenne).
   - **Période** : **Samedi 3 octobre à 15h00** (proche de la date actuelle, donc idéal si vous cherchez une activité pour ces prochains jours).

2. **"BISFF in Paris - Conférence : L’IA et le court-métrage coréen"** :
   - **Description** : Une conférence intéressante qui explore l'intersection entre l'innovation technologique coréenne et l'art cinématographique français, avec une réflexion sur l'impact de l'IA dans le cinéma.
   - **Lieu** : Centre Culturel Coréen (20 Rue La Boétie, 75008 Paris).
   - **Période** : **Lundi 12 octobre à 19h00** (un peu plus loin dans le temps, mais une option très originale et intellectuel